In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Set visual style
plt.style.use('ggplot')

def create_sample_dataset(filepath='/workspace/scratch/sample_ml_dataset.csv'):
    """Generates a synthetic binary classification dataset for demonstration."""
    np.random.seed(42)
    n_samples = 600

    # Ensure the directory exists
    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    age = np.random.randint(18, 65, size=n_samples)
    income = np.random.randint(20000, 120000, size=n_samples)
    credit_score = np.random.randint(300, 850, size=n_samples)
    years_employed = np.random.randint(0, 30, size=n_samples)

    # Target probability centered around 0
    logit = (age - 40) * 0.04 + (income - 60000) * 0.00004 + (credit_score - 600) * 0.007 + (years_employed - 10) * 0.08 + np.random.normal(0, 1, n_samples)
    prob = 1 / (1 + np.exp(-logit))
    target = (prob > 0.5).astype(int)

    df = pd.DataFrame({
        'Age': age,
        'Income': income,
        'Credit_Score': credit_score,
        'Years_Employed': years_employed,
        'Target': target
    })

    df.to_csv(filepath, index=False)
    print(f"Sample dataset saved to '{filepath}'.")
    return filepath

def run_multi_model_comparison(csv_path, target_col='Target'):
    print("=== STEP 1: LOADING & INSPECTING DATASET ===")
    df = pd.read_csv(csv_path)
    print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print("\nClass Distribution:")
    print(df[target_col].value_counts(normalize=True))


    # STEP 2: PREPARE FEATURES AND TARGET

    print("\n=== STEP 2: TRAIN-TEST SPLIT & SCALING ===")
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Split dataset into 80% training and 20% testing
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Testing set: {X_test.shape[0]} samples")

    # Feature Scaling (Crucial for Logistic Regression)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    # STEP 3: TRAIN MULTIPLE MODELS

    print("\n=== STEP 3: TRAINING MULTIPLE ALGORITHMS ===")
    models = {
        'Logistic Regression': (LogisticRegression(random_state=42), X_train_scaled, X_test_scaled),
        'Decision Tree': (DecisionTreeClassifier(max_depth=5, random_state=42), X_train, X_test),
        'Random Forest': (RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42), X_train, X_test)
    }

    results = []
    roc_data = {}
    confusion_matrices = {}

    for name, (model, X_tr, X_te) in models.items():
        # Train model
        model.fit(X_tr, y_train)

        # Predictions & Probabilities
        y_pred = model.predict(X_te)
        y_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else y_pred

        # Metrics calculation
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)

        results.append({
            'Model': name,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-Score': f1,
            'ROC-AUC': auc
        })

        # Store ROC curve points and confusion matrix
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_data[name] = (fpr, tpr, auc)
        confusion_matrices[name] = confusion_matrix(y_test, y_pred)

    # Convert results to DataFrame for clean presentation
    comparison_df = pd.DataFrame(results)
    print("\n=== MODEL COMPARISON TABLE ===")
    print(comparison_df.to_string(index=False))

    # Identify best model based on ROC-AUC
    best_model_row = comparison_df.loc[comparison_df['ROC-AUC'].idxmax()]
    print(f"\n🏆 Best Model based on ROC-AUC: {best_model_row['Model']} (AUC = {best_model_row['ROC-AUC']:.4f})")


    # STEP 4: VISUALIZATIONS

    print("\n=== STEP 4: GENERATING COMPARISON PLOTS ===")

    # Ensure the directory exists before saving plots
    plot_dir = '/workspace/scratch'
    os.makedirs(plot_dir, exist_ok=True)

    # 1. Plot Combined ROC Curves
    plt.figure(figsize=(8, 6))
    colors = {'Logistic Regression': 'darkorange', 'Decision Tree': 'green', 'Random Forest': 'purple'}

    for name, (fpr, tpr, auc) in roc_data.items():
        plt.plot(fpr, tpr, color=colors[name], lw=2, label=f'{name} (AUC = {auc:.3f})')

    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess (AUC = 0.50)')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
    plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
    plt.title('Model Comparison - ROC Curves', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, 'model_comparison_roc.png'), dpi=150)
    plt.close()
    print(f"Saved '{os.path.join(plot_dir, 'model_comparison_roc.png')}'")

    # 2. Plot Side-by-Side Confusion Matrices
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, (name, cm) in zip(axes, confusion_matrices.items()):
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                    xticklabels=['Neg (0)', 'Pos (1)'],
                    yticklabels=['Neg (0)', 'Pos (1)'])
        ax.set_title(f"{name}\nConfusion Matrix", fontsize=12, fontweight='bold')
        ax.set_xlabel("Predicted Label")
        ax.set_ylabel("True Label")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, 'model_comparison_confusion_matrices.png'), dpi=150)
    plt.close()
    print(f"Saved '{os.path.join(plot_dir, 'model_comparison_confusion_matrices.png')}'")

if __name__ == '__main__':
    data_file = create_sample_dataset()
    run_multi_model_comparison(data_file)

Sample dataset saved to '/workspace/scratch/sample_ml_dataset.csv'.
=== STEP 1: LOADING & INSPECTING DATASET ===
Dataset Shape: 600 rows, 5 columns

Class Distribution:
Target
1    0.586667
0    0.413333
Name: proportion, dtype: float64

=== STEP 2: TRAIN-TEST SPLIT & SCALING ===
Training set: 480 samples
Testing set: 120 samples

=== STEP 3: TRAINING MULTIPLE ALGORITHMS ===

=== MODEL COMPARISON TABLE ===
              Model  Accuracy  Precision   Recall  F1-Score  ROC-AUC
Logistic Regression  0.833333   0.847222 0.871429  0.859155 0.933143
      Decision Tree  0.866667   0.885714 0.885714  0.885714 0.887143
      Random Forest  0.816667   0.815789 0.885714  0.849315 0.923429

🏆 Best Model based on ROC-AUC: Logistic Regression (AUC = 0.9331)

=== STEP 4: GENERATING COMPARISON PLOTS ===
Saved '/workspace/scratch/model_comparison_roc.png'
Saved '/workspace/scratch/model_comparison_confusion_matrices.png'
